# Faceoff Decay Analysis (Phase 2, Area 3)

**Goal:** Determine how shot quality decays as time elapses after a faceoff.

**Key questions:**
1. Do the proposed recency bin boundaries (5/15/30/60s) match inflection points in the data?
2. Do decay patterns differ by faceoff zone (O/D/N)?
3. Does manpower state interact with faceoff decay?
4. Is a continuous representation (e.g., `log(seconds + 1)`) better than bins?

**Data source:** `shot_events` table in `data/nhl_data.db`

In [ ]:
import os
import sys
import sqlite3

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# Add src/ to path — handle CWD being project root or notebooks/
for _candidate in [os.path.join(os.getcwd(), "src"),
                   os.path.join(os.getcwd(), "..", "src")]:
    _candidate = os.path.abspath(_candidate)
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from database import DATABASE_PATH

sns.set_theme(style="whitegrid")
print(f"Database: {DATABASE_PATH}")
conn = sqlite3.connect(DATABASE_PATH)
conn.row_factory = sqlite3.Row
print("Connected.")

## 1. Data availability check

Verify we have enough shot events with `seconds_since_faceoff` populated.

In [ ]:
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM shot_events")
total_shots = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM shot_events WHERE seconds_since_faceoff IS NOT NULL")
with_faceoff = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM shot_events WHERE faceoff_zone_code IS NOT NULL")
with_zone = cur.fetchone()[0]

coverage_pct = (with_faceoff / total_shots * 100) if total_shots > 0 else 0

print(f"Total shot events:               {total_shots:,}")
print(f"With seconds_since_faceoff:       {with_faceoff:,} ({coverage_pct:.1f}%)")
print(f"With faceoff_zone_code:           {with_zone:,}")

if total_shots == 0:
    print("\n*** No shot events found. Run the scraper first. ***")

## 2. Shot rate and goal rate by 1-second buckets

The core decay curve: how does shot volume and conversion rate change as seconds elapse after a faceoff? Capped at 120 seconds (beyond that, faceoff context is negligible).

In [ ]:
MAX_SECONDS = 120

cur.execute("""
    SELECT seconds_since_faceoff AS sec,
           COUNT(*)              AS shot_count,
           SUM(is_goal)          AS goal_count
    FROM shot_events
    WHERE seconds_since_faceoff IS NOT NULL
      AND seconds_since_faceoff BETWEEN 0 AND ?
    GROUP BY seconds_since_faceoff
    ORDER BY seconds_since_faceoff
""", (MAX_SECONDS,))

rows = cur.fetchall()
seconds = np.array([r[0] for r in rows])
shot_counts = np.array([r[1] for r in rows])
goal_counts = np.array([r[2] for r in rows])
goal_rate = np.where(shot_counts > 0, goal_counts / shot_counts, 0.0)

print(f"Seconds covered: {len(seconds)} buckets, {shot_counts.sum():,} shots, {goal_counts.sum():,} goals")

In [ ]:
# Proposed bin boundaries
BIN_BOUNDARIES = [5, 15, 30, 60]
BIN_COLORS = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71"]
BIN_LABELS = ["immediate\n(0-5s)", "early\n(6-15s)", "mid\n(16-30s)", "late\n(31-60s)"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Shot count by second
ax1.bar(seconds, shot_counts, width=1.0, color="#3498db", alpha=0.7, label="Shots")
ax1.set_ylabel("Shot count")
ax1.set_title("Shot Volume by Seconds Since Faceoff")
for boundary, color, label in zip(BIN_BOUNDARIES, BIN_COLORS, BIN_LABELS):
    ax1.axvline(x=boundary + 0.5, color=color, linestyle="--", linewidth=1.5,
                label=f"{label.split(chr(10))[0]} boundary ({boundary}s)")
ax1.legend(loc="upper right", fontsize=8)

# Goal rate by second (smoothed with 5-second rolling average)
SMOOTH_WINDOW = 5
if len(goal_rate) >= SMOOTH_WINDOW:
    smoothed = np.convolve(goal_rate, np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW, mode="same")
else:
    smoothed = goal_rate

ax2.plot(seconds, goal_rate, alpha=0.3, color="#95a5a6", linewidth=0.8, label="Raw (1s)")
ax2.plot(seconds, smoothed, color="#e74c3c", linewidth=2.0, label=f"Smoothed ({SMOOTH_WINDOW}s avg)")
ax2.set_ylabel("Goal rate (goals / shots)")
ax2.set_xlabel("Seconds since faceoff")
ax2.set_title("Goal Rate by Seconds Since Faceoff")
for boundary, color in zip(BIN_BOUNDARIES, BIN_COLORS):
    ax2.axvline(x=boundary + 0.5, color=color, linestyle="--", linewidth=1.5)
ax2.legend(loc="upper right", fontsize=8)

fig.tight_layout()
plt.show()

## 3. Aggregate stats by proposed recency bin

Summarize shot volume and goal rate within each proposed bin to evaluate whether the boundaries capture meaningful differences.

In [ ]:
cur.execute("""
    SELECT
        CASE
            WHEN seconds_since_faceoff BETWEEN 0 AND 5   THEN 'immediate (0-5s)'
            WHEN seconds_since_faceoff BETWEEN 6 AND 15  THEN 'early (6-15s)'
            WHEN seconds_since_faceoff BETWEEN 16 AND 30 THEN 'mid (16-30s)'
            WHEN seconds_since_faceoff BETWEEN 31 AND 60 THEN 'late (31-60s)'
            ELSE 'steady_state (61+s)'
        END AS recency_bin,
        COUNT(*)                                           AS shots,
        SUM(is_goal)                                       AS goals,
        ROUND(CAST(SUM(is_goal) AS REAL) / COUNT(*), 4)    AS goal_rate,
        ROUND(AVG(distance_to_goal), 1)                    AS avg_distance
    FROM shot_events
    WHERE seconds_since_faceoff IS NOT NULL
      AND seconds_since_faceoff >= 0
    GROUP BY recency_bin
    ORDER BY
        CASE recency_bin
            WHEN 'immediate (0-5s)' THEN 1
            WHEN 'early (6-15s)' THEN 2
            WHEN 'mid (16-30s)' THEN 3
            WHEN 'late (31-60s)' THEN 4
            ELSE 5
        END
""")

print(f"{'Recency Bin':<25} {'Shots':>10} {'Goals':>8} {'Goal Rate':>10} {'Avg Dist':>10}")
print("-" * 68)
for row in cur.fetchall():
    print(f"{row[0]:<25} {row[1]:>10,} {row[2]:>8,} {row[3]:>10.4f} {row[4]:>10.1f}")

## 4. Decay curves by faceoff zone

Split the decay curve by `faceoff_zone_code` (O = offensive, D = defensive, N = neutral). Offensive-zone faceoffs should show a sharper initial spike in shot quality.

In [ ]:
ZONE_CODES = ["O", "D", "N"]
ZONE_LABELS = {"O": "Offensive zone", "D": "Defensive zone", "N": "Neutral zone"}
ZONE_COLORS = {"O": "#e74c3c", "D": "#3498db", "N": "#2ecc71"}

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for zone in ZONE_CODES:
    cur.execute("""
        SELECT seconds_since_faceoff,
               COUNT(*) AS shots,
               SUM(is_goal) AS goals
        FROM shot_events
        WHERE seconds_since_faceoff IS NOT NULL
          AND seconds_since_faceoff BETWEEN 0 AND ?
          AND faceoff_zone_code = ?
        GROUP BY seconds_since_faceoff
        ORDER BY seconds_since_faceoff
    """, (MAX_SECONDS, zone))

    rows = cur.fetchall()
    if not rows:
        continue

    z_sec = np.array([r[0] for r in rows])
    z_shots = np.array([r[1] for r in rows])
    z_goals = np.array([r[2] for r in rows])
    z_rate = np.where(z_shots > 0, z_goals / z_shots, 0.0)

    # Smooth
    if len(z_rate) >= SMOOTH_WINDOW:
        z_smooth = np.convolve(z_rate, np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW, mode="same")
    else:
        z_smooth = z_rate

    color = ZONE_COLORS[zone]
    label = ZONE_LABELS[zone]

    axes[0].plot(z_sec, z_shots, color=color, alpha=0.8, linewidth=1.5, label=label)
    axes[1].plot(z_sec, z_smooth, color=color, linewidth=2.0, label=label)

axes[0].set_ylabel("Shot count")
axes[0].set_title("Shot Volume by Zone and Seconds Since Faceoff")
axes[0].legend()

axes[1].set_ylabel("Goal rate (smoothed)")
axes[1].set_xlabel("Seconds since faceoff")
axes[1].set_title("Goal Rate by Zone and Seconds Since Faceoff")
axes[1].legend()

for ax in axes:
    for boundary in BIN_BOUNDARIES:
        ax.axvline(x=boundary + 0.5, color="#bdc3c7", linestyle=":", linewidth=1.0)

fig.tight_layout()
plt.show()

## 5. Zone-by-bin aggregate table

Cross-tabulate zone and recency bin to see the interaction numerically.

In [ ]:
cur.execute("""
    SELECT
        faceoff_zone_code AS zone,
        CASE
            WHEN seconds_since_faceoff BETWEEN 0 AND 5   THEN '1_immediate'
            WHEN seconds_since_faceoff BETWEEN 6 AND 15  THEN '2_early'
            WHEN seconds_since_faceoff BETWEEN 16 AND 30 THEN '3_mid'
            WHEN seconds_since_faceoff BETWEEN 31 AND 60 THEN '4_late'
            ELSE '5_steady_state'
        END AS recency_bin,
        COUNT(*)                                        AS shots,
        SUM(is_goal)                                    AS goals,
        ROUND(CAST(SUM(is_goal) AS REAL) / COUNT(*), 4) AS goal_rate,
        ROUND(AVG(distance_to_goal), 1)                 AS avg_dist
    FROM shot_events
    WHERE seconds_since_faceoff IS NOT NULL
      AND seconds_since_faceoff >= 0
      AND faceoff_zone_code IS NOT NULL
    GROUP BY zone, recency_bin
    ORDER BY zone, recency_bin
""")

print(f"{'Zone':<6} {'Bin':<18} {'Shots':>10} {'Goals':>8} {'Goal Rate':>10} {'Avg Dist':>10}")
print("-" * 66)
for row in cur.fetchall():
    bin_label = row[1].split("_", 1)[1]
    print(f"{row[0]:<6} {bin_label:<18} {row[2]:>10,} {row[3]:>8,} {row[4]:>10.4f} {row[5]:>10.1f}")

## 6. Decay by manpower state

Does the post-faceoff spike differ on the power play vs even strength? PP faceoffs in the offensive zone may produce a stronger immediate scoring spike.

In [ ]:
# Group manpower states into simplified categories for plotting
MANPOWER_GROUPS = {
    "Even strength": ("5v5", "4v4", "3v3"),
    "Power play":    ("5v4", "5v3", "4v3", "6v5", "6v4", "6v3"),
    "Short-handed":  ("4v5", "3v5", "3v4", "5v6", "4v6", "3v6"),
}
MP_COLORS = {"Even strength": "#3498db", "Power play": "#e74c3c", "Short-handed": "#2ecc71"}

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for mp_label, mp_states in MANPOWER_GROUPS.items():
    placeholders = ",".join(["?"] * len(mp_states))
    cur.execute(f"""
        SELECT seconds_since_faceoff,
               COUNT(*) AS shots,
               SUM(is_goal) AS goals
        FROM shot_events
        WHERE seconds_since_faceoff IS NOT NULL
          AND seconds_since_faceoff BETWEEN 0 AND ?
          AND manpower_state IN ({placeholders})
        GROUP BY seconds_since_faceoff
        ORDER BY seconds_since_faceoff
    """, (MAX_SECONDS, *mp_states))

    rows = cur.fetchall()
    if not rows:
        continue

    m_sec = np.array([r[0] for r in rows])
    m_shots = np.array([r[1] for r in rows])
    m_goals = np.array([r[2] for r in rows])
    m_rate = np.where(m_shots > 0, m_goals / m_shots, 0.0)

    if len(m_rate) >= SMOOTH_WINDOW:
        m_smooth = np.convolve(m_rate, np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW, mode="same")
    else:
        m_smooth = m_rate

    color = MP_COLORS[mp_label]
    axes[0].plot(m_sec, m_shots, color=color, alpha=0.8, linewidth=1.5, label=mp_label)
    axes[1].plot(m_sec, m_smooth, color=color, linewidth=2.0, label=mp_label)

axes[0].set_ylabel("Shot count")
axes[0].set_title("Shot Volume by Manpower State and Seconds Since Faceoff")
axes[0].legend()

axes[1].set_ylabel("Goal rate (smoothed)")
axes[1].set_xlabel("Seconds since faceoff")
axes[1].set_title("Goal Rate by Manpower State and Seconds Since Faceoff")
axes[1].legend()

for ax in axes:
    for boundary in BIN_BOUNDARIES:
        ax.axvline(x=boundary + 0.5, color="#bdc3c7", linestyle=":", linewidth=1.0)

fig.tight_layout()
plt.show()

## 7. Continuous vs. binned vs. log-transform comparison

If the empirical decay curve is roughly exponential, `log(seconds_since_faceoff + 1)` may outperform categorical bins. Compare the shape of the raw data to an exponential fit and a log transform.

In [ ]:
# Use shot counts (proxy for "shot generation rate") as the decay signal
# Normalize to [0,1] relative to max for comparison
if len(shot_counts) > 0 and shot_counts.max() > 0:
    normalized_counts = shot_counts / shot_counts.max()

    # Log transform of seconds
    log_seconds = np.log1p(seconds)  # log(seconds + 1)

    # Fit exponential: count ~ A * exp(-lambda * t)
    # Use log-linear regression on the normalized counts
    valid = normalized_counts > 0
    if valid.sum() > 2:
        log_counts = np.log(normalized_counts[valid])
        sec_valid = seconds[valid]

        # Simple OLS: log(y) = a + b*t => y = exp(a) * exp(b*t)
        coeffs = np.polyfit(sec_valid, log_counts, 1)
        decay_rate = coeffs[0]
        amplitude = np.exp(coeffs[1])
        exp_fit = amplitude * np.exp(decay_rate * seconds)

        fig, axes = plt.subplots(1, 3, figsize=(16, 5))

        # Panel 1: Raw counts + exponential fit
        axes[0].bar(seconds, normalized_counts, width=1.0, alpha=0.5, color="#3498db", label="Observed")
        axes[0].plot(seconds, exp_fit, color="#e74c3c", linewidth=2.0,
                     label=f"Exp fit (decay={decay_rate:.4f}/s)")
        axes[0].set_xlabel("Seconds since faceoff")
        axes[0].set_ylabel("Normalized shot count")
        axes[0].set_title("Exponential decay fit")
        axes[0].legend(fontsize=8)

        # Panel 2: Log(seconds+1) vs goal rate
        axes[1].scatter(log_seconds, goal_rate, alpha=0.3, s=8, color="#3498db")
        if len(goal_rate) >= SMOOTH_WINDOW:
            # Sort by log_seconds and smooth
            sort_idx = np.argsort(log_seconds)
            axes[1].plot(log_seconds[sort_idx], smoothed[sort_idx],
                         color="#e74c3c", linewidth=2.0, label="Smoothed")
        axes[1].set_xlabel("log(seconds + 1)")
        axes[1].set_ylabel("Goal rate")
        axes[1].set_title("Goal rate vs log-transformed time")
        axes[1].legend(fontsize=8)

        # Panel 3: Residual of exponential fit
        residuals = normalized_counts - exp_fit
        axes[2].bar(seconds, residuals, width=1.0, alpha=0.6, color="#2ecc71")
        axes[2].axhline(y=0, color="black", linewidth=0.5)
        axes[2].set_xlabel("Seconds since faceoff")
        axes[2].set_ylabel("Residual (observed - fit)")
        axes[2].set_title("Exponential fit residuals")

        fig.tight_layout()
        plt.show()

        # Summary statistics
        ss_res = np.sum(residuals ** 2)
        ss_tot = np.sum((normalized_counts - normalized_counts.mean()) ** 2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        print(f"Exponential fit R-squared: {r_squared:.4f}")
        print(f"Decay rate: {decay_rate:.5f} per second (half-life: {-np.log(2)/decay_rate:.1f}s)")
    else:
        print("Not enough data points for exponential fit.")
else:
    print("No shot count data available.")

## 8. Average shot distance by recency

If shots taken immediately after faceoffs are closer to the net (lower distance), that helps explain the goal rate spike independent of shot volume.

In [ ]:
cur.execute("""
    SELECT seconds_since_faceoff,
           AVG(distance_to_goal) AS avg_dist,
           COUNT(*) AS n
    FROM shot_events
    WHERE seconds_since_faceoff IS NOT NULL
      AND seconds_since_faceoff BETWEEN 0 AND ?
      AND distance_to_goal IS NOT NULL
    GROUP BY seconds_since_faceoff
    ORDER BY seconds_since_faceoff
""", (MAX_SECONDS,))

rows = cur.fetchall()
d_sec = np.array([r[0] for r in rows])
d_avg = np.array([r[1] for r in rows])

if len(d_avg) >= SMOOTH_WINDOW:
    d_smooth = np.convolve(d_avg, np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW, mode="same")
else:
    d_smooth = d_avg

fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(d_sec, d_avg, alpha=0.3, s=10, color="#95a5a6", label="Raw (1s)")
ax.plot(d_sec, d_smooth, color="#8e44ad", linewidth=2.0, label=f"Smoothed ({SMOOTH_WINDOW}s avg)")
ax.set_xlabel("Seconds since faceoff")
ax.set_ylabel("Average distance to goal (ft)")
ax.set_title("Shot Distance by Seconds Since Faceoff")
for boundary in BIN_BOUNDARIES:
    ax.axvline(x=boundary + 0.5, color="#bdc3c7", linestyle=":", linewidth=1.0)
ax.legend()
fig.tight_layout()
plt.show()

## 9. Summary and recommendations

**Interpret the plots above to answer:**

1. **Bin boundaries:** Do the vertical dashed lines at 5/15/30/60s align with visible inflection points in the shot volume and goal rate curves? If not, adjust `_FACEOFF_RECENCY_BINS` in `xg_features.py`.

2. **Zone interaction:** Is the decay pattern meaningfully different across O/D/N zones? If offensive zone shows a much sharper spike, the `faceoff_zone_recency_interaction` feature is justified. If all zones decay similarly, zone code alone (already in `shot_events`) may suffice.

3. **Manpower interaction:** Does PP show a different decay shape than even strength? If so, consider interacting faceoff recency with manpower state in the xG model (Phase 3).

4. **Continuous vs. binned:** If the exponential fit R-squared is high (>0.8) and residuals are small, `log(seconds_since_faceoff + 1)` may outperform bins. If there are clear step-function breaks, bins are better. A pragmatic approach: include both and let the model select.

In [ ]:
conn.close()
print("Done.")